<a href="https://colab.research.google.com/github/preethim528-arch/scamshield-gemini-ai/blob/main/scamshield_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
# 1. Install required packages
!pip install -q -U google-genai gradio gTTS pillow

import gradio as gr
from google import genai
from google.genai import types
import json
from gtts import gTTS
import os

# 2. Put your actual Gemini API Key here
GEMINI_API_KEY = "YOUR_ACTUAL_API_KEY_HERE"
client = genai.Client(api_key=GEMINI_API_KEY)

# 3. System Prompt for AI
SYSTEM_PROMPT = """
You are ScamShield, an advanced cybersecurity AI assistant for non-tech users.
Analyze the input text or screenshot. Output MUST be strict JSON format:
{
  "status": "SAFE" or "WARNING" or "DANGER",
  "score": (integer 0 to 100),
  "category": "e.g., Electricity Bill / Phishing / OTP Trap / Legitimate",
  "tamil_voice_script": "2 short lines in spoken Tamil explaining if it is safe or scam, and what action to take.",
  "action_english": "e.g., DO NOT CLICK / DELETE MESSAGE / SAFE TRANSACTION",
  "key_flags": ["Reason 1", "Reason 2"]
}
"""

# Custom CSS for Sleek Glassmorphism Dashboard
CUSTOM_CSS = """
.gradio-container {
    max-width: 1050px !important;
    margin: auto !important;
    font-family: 'Segoe UI', system-ui, -apple-system, sans-serif !important;
}
.hero-card {
    background: linear-gradient(135deg, #1e1e38 0%, #0f172a 100%);
    border-radius: 16px;
    padding: 24px;
    color: white;
    box-shadow: 0 10px 25px rgba(0,0,0,0.25);
    border: 1px solid rgba(255, 255, 255, 0.1);
    margin-bottom: 15px;
}
.stat-pill {
    background: rgba(255, 255, 255, 0.08);
    border-radius: 12px;
    padding: 12px 18px;
    border: 1px solid rgba(255, 255, 255, 0.12);
    text-align: center;
}
.cyber-card {
    background: #ffffff;
    border-radius: 16px;
    padding: 20px;
    box-shadow: 0 8px 30px rgba(0,0,0,0.06);
    border: 1px solid #e2e8f0;
}
"""

def scan_threat(image, text):
    if not image and not text:
        return (
            "<div style='color: #ef4444; font-weight: bold;'>⚠️ Please enter SMS text or upload a screenshot!</div>",
            "",
            None
        )

    contents = []
    if image:
        contents.append(image)
    if text:
        contents.append(text)

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=contents,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                response_mime_type="application/json",
                temperature=0.1
            )
        )
        data = json.loads(response.text)

        status = data.get("status", "SAFE")
        score = data.get("score", 0)
        category = data.get("category", "General Verification")
        action = data.get("action_english", "No Action Required")
        flags = data.get("key_flags", [])
        tamil_msg = data.get("tamil_voice_script", "Endha aabathum illai.")

        # Color Palettes & Status Badges
        if status == "DANGER":
            theme_color = "#ef4444"
            bg_color = "#fef2f2"
            icon = "🚨"
            status_text = "CRITICAL THREAT / மோசடி"
        elif status == "WARNING":
            theme_color = "#f59e0b"
            bg_color = "#fffbeb"
            icon = "⚠️"
            status_text = "SUSPICIOUS / எச்சரிக்கை"
        else:
            theme_color = "#10b981"
            bg_color = "#f0fdf4"
            icon = "✅"
            status_text = "SAFE / பாதுகாப்பானது"

        flags_html = "".join([f"<li style='margin-bottom: 4px;'>{f}</li>" for f in flags])

        # Visual Status Card
        badge_html = f"""
        <div style='background: {bg_color}; border-left: 6px solid {theme_color}; border-radius: 12px; padding: 20px; box-shadow: 0 4px 15px rgba(0,0,0,0.05);'>
            <div style='display: flex; justify-content: space-between; align-items: center;'>
                <div>
                    <span style='font-size: 24px; vertical-align: middle;'>{icon}</span>
                    <strong style='font-size: 18px; color: {theme_color}; margin-left: 8px;'>{status_text}</strong>
                </div>
                <div style='background: {theme_color}; color: white; border-radius: 20px; padding: 4px 14px; font-weight: bold; font-size: 14px;'>
                    Risk Score: {score}%
                </div>
            </div>
            <div style='margin-top: 10px; color: #64748b; font-size: 13px;'>
                <strong>Category:</strong> {category}
            </div>
        </div>
        """

        # Helpline Card for Danger
        helpline_html = ""
        if score >= 70:
            helpline_html = """
            <div style='margin-top: 15px; background: #fee2e2; border-radius: 10px; padding: 12px; border: 1px dashed #ef4444; color: #991b1b;'>
                🛡️ <strong>National Cyber Helpline:</strong> Call <strong>1930</strong> or report at <a href='https://cybercrime.gov.in' target='_blank' style='color:#dc2626; font-weight:bold;'>cybercrime.gov.in</a>
            </div>
            """

        # Detailed Analytics Card
        summary_html = f"""
        <div style='background: white; border-radius: 12px; padding: 20px; border: 1px solid #e2e8f0;'>
            <h4 style='margin-top:0; color: #1e293b;'>📋 Recommended Action</h4>
            <div style='font-size: 16px; font-weight: bold; color: {theme_color}; background: #f8fafc; padding: 10px 14px; border-radius: 8px;'>
                👉 {action}
            </div>

            <h4 style='margin-top: 18px; margin-bottom: 8px; color: #1e293b;'>🔍 Risk Indicators:</h4>
            <ul style='color: #475569; padding-left: 20px; font-size: 14px;'>
                {flags_html if flags_html else '<li>No immediate malicious indicators detected.</li>'}
            </ul>

            <h4 style='margin-top: 18px; margin-bottom: 8px; color: #1e293b;'>🎙️ Tamil Voice Brief:</h4>
            <div style='font-size: 14px; color: #334155; font-style: italic; background: #f1f5f9; padding: 10px 14px; border-radius: 8px;'>
                "{tamil_msg}"
            </div>
            {helpline_html}
        </div>
        """

        # Audio generation
        tts = gTTS(text=tamil_msg, lang='ta')
        audio_file = "alert_voice.mp3"
        tts.save(audio_file)

        return badge_html, summary_html, audio_file

    except Exception as e:
        return f"<div style='color: red;'>❌ Error: {str(e)}</div>", "", None

# 4. Professional Gradio Dashboard Layout
with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo"), css=CUSTOM_CSS) as app:

    # Hero Header Section
    gr.HTML("""
    <div class='hero-card'>
        <div style='display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap;'>
            <div>
                <h1 style='margin: 0; font-size: 26px; font-weight: 800; color: #ffffff;'>🛡️ ScamShield AI Dashboard</h1>
                <p style='margin: 6px 0 0 0; color: #cbd5e1; font-size: 14px;'>Multimodal threat detection & automated Tamil voice assist for non-tech users.</p>
            </div>
            <div style='display: flex; gap: 12px; margin-top: 10px;'>
                <div class='stat-pill'>
                    <div style='font-size: 11px; color: #94a3b8; text-transform: uppercase;'>Engine</div>
                    <div style='font-size: 14px; font-weight: bold; color: #38bdf8;'>Gemini 2.5 Flash</div>
                </div>
                <div class='stat-pill'>
                    <div style='font-size: 11px; color: #94a3b8; text-transform: uppercase;'>Voice Support</div>
                    <div style='font-size: 14px; font-weight: bold; color: #4ade80;'>Tamil (gTTS)</div>
                </div>
            </div>
        </div>
    </div>
    """)

    with gr.Row():
        # Left Panel: User Inputs
        with gr.Column(scale=5):
            gr.HTML("<h3 style='margin-bottom: 8px; color: #334155;'>📥 Inspect Incoming Threat</h3>")
            img = gr.Image(type="pil", label="Upload WhatsApp / Payment Screenshot", height=240)
            txt = gr.Textbox(placeholder="Or paste SMS, email, or WhatsApp text here...", label="Threat Message Content", lines=4)
            btn = gr.Button("🛡️ Analyze Threat Level", variant="primary", size="lg")

        # Right Panel: Security Output
        with gr.Column(scale=6):
            gr.HTML("<h3 style='margin-bottom: 8px; color: #334155;'>📊 Threat Intelligence & Audio</h3>")
            status_box = gr.HTML(label="Risk Status")
            details_box = gr.HTML(label="Detailed Analysis")
            audio_box = gr.Audio(label="🔊 Tamil Voice Advisory", type="filepath")

    # Wire event
    btn.click(fn=scan_threat, inputs=[img, txt], outputs=[status_box, details_box, audio_box])

# 5. Launch App
app.launch(share=True)

/tmp/ipykernel_562/2160952725.py:172: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo"), css=CUSTOM_CSS) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3946e81d1ca2f0e654.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
